# Sesión 6 - Ejercicios: DDL y DML con PostgreSQL

Trabajamos con las mismas tablas del registro escolar ficticio, ahora con permisos
de escritura sobre nuestro schema personal en Supabase.

Ejecuta las celdas en orden. Cada ejercicio construye sobre el anterior.

## Configuración de la conexión

Instalamos `psycopg2-binary` y `polars`, establecemos la conexión y definimos la función `sql()`
que ejecuta cualquier consulta — tanto `SELECT` como DDL y DML.

Reemplaza `USUARIO_BASE` y `TU_ESQUEMA` con tus datos (ver `usuarios_sql.csv`).

In [ ]:
%pip install psycopg2-binary polars --quiet

In [ ]:
import psycopg2
import polars as pl

USUARIO_BASE = "COMPLETAR"   # tu usuario, ver usuarios_sql.csv
PROYECTO     = "jcgopqiutpioycndzkdg"
PASSWORD     = "flacso_eseda"
HOST         = "aws-1-sa-east-1.pooler.supabase.com"
PORT         = 6543
DB           = "postgres"

conn = psycopg2.connect(
    host=HOST,
    port=PORT,
    dbname=DB,
    user=f"{USUARIO_BASE}.{PROYECTO}",
    password=PASSWORD,
    sslmode="require",
)
conn.autocommit = True

def sql(query):
    """Ejecuta una query. Devuelve DataFrame si hay filas, None si es DDL/DML."""
    with conn.cursor() as cur:
        cur.execute(query)
        if cur.description:
            cols = [desc[0] for desc in cur.description]
            rows = cur.fetchall()
            return pl.DataFrame(rows, schema=cols, orient="row")
        return None

print("Conexión exitosa")

## Ejercicio 0 - Intro: permisos y schemas

En la sesión anterior tuvimos acceso de solo lectura sobre el schema `public`.
En esta sesión cada quien tiene su propio schema con permisos completos.

Primero comprobamos qué pasa cuando intentamos escribir sobre `public`.

In [ ]:
# Intentar insertar en public.alumnos
# Resultado esperado: ERROR de permisos
sql("""
INSERT INTO public.alumnos (cod_estudiante, nombre, sexo, edad, pais, ciudad,
                            altura, paralelo, anio_nac, mes_nac, dia_nac)
VALUES (999, 'Prueba Permiso', 'm', 25, 'Ecuador', 'Quito', 170, 'A', 1999, 1, 1);
""")

In [ ]:
# Verificar cuál es nuestro schema personal
sql("""
SELECT schema_name
FROM   information_schema.schemata
WHERE  schema_name NOT IN ('public', 'pg_catalog', 'information_schema',
                           'pg_toast', 'pg_temp_1', 'pg_toast_temp_1')
  AND  schema_name NOT LIKE 'pg_%'
ORDER  BY schema_name;
""")

## Ejercicio 1 - DDL: CREATE TABLE, ALTER TABLE, DROP TABLE y SET search_path

Creamos tablas en nuestro schema personal, inspeccionamos los objetos creados
con `information_schema`, y simplificamos el trabajo con `SET search_path`.

In [ ]:
# Crear una tabla de prueba con nombre temporal
sql("""
CREATE TABLE <tu_esquema>.notas_123 (
    cod_estudiante  INTEGER,
    nombre          TEXT,
    materia         TEXT,
    nota            NUMERIC
);
""")

In [ ]:
# Verificar que la tabla existe en nuestro schema
sql("""
SELECT table_name
FROM   information_schema.tables
WHERE  table_schema = '<tu_esquema>'
ORDER  BY table_name;
""")

In [ ]:
# Renombrar la tabla a su nombre definitivo
sql("""
ALTER TABLE <tu_esquema>._________ RENAME TO notas;
""")

In [ ]:
# Verificar el cambio de nombre
sql("""
SELECT table_name
FROM   information_schema.tables
WHERE  table_schema = '<tu_esquema>'
ORDER  BY table_name;
""")

In [ ]:
# Crear la tabla alumnos con PRIMARY KEY y CHECK
sql("""
CREATE TABLE <tu_esquema>.alumnos (
    cod_estudiante  INTEGER       ______________,
    nombre          TEXT          ______________,
    sexo            TEXT          NOT NULL CHECK (____________),
    edad            INTEGER       CHECK (edad >= 0 AND edad <= 120),
    pais            TEXT,
    ciudad          TEXT,
    altura          NUMERIC,
    paralelo        TEXT,
    anio_nac        _______,
    mes_nac         _______,
    dia_nac         _______
);
""")

In [ ]:
# Verificar columnas y tipos de alumnos
sql("""
SELECT column_name,
       data_type,
       is_nullable
FROM   information_schema.columns
WHERE  table_schema = '<tu_esquema>'
  AND  table_name   = 'alumnos'
ORDER  BY ordinal_position;
""")

In [ ]:
# Ver las restricciones definidas en alumnos
sql("""
SELECT constraint_name,
       constraint_type
FROM   information_schema.table_constraints
WHERE  table_schema = '<tu_esquema>'
  AND  table_name   = 'alumnos'
ORDER  BY constraint_type;
""")

In [ ]:
# Agregar la columna email con ALTER TABLE
sql("""
ALTER TABLE <tu_esquema>._______
    ADD COLUMN ________ ______;
""")

In [ ]:
# Verificar que la columna fue agregada
sql("""
SELECT column_name, data_type
FROM   information_schema.columns
WHERE  table_schema = '<tu_esquema>'
  AND  table_name   = 'alumnos'
ORDER  BY ordinal_position;
""")

In [ ]:
# Eliminar la columna email
sql("""
ALTER TABLE <tu_esquema>.________
    DROP COLUMN _______;
""")

In [ ]:
# Crear tabla de prueba y eliminarla con DROP
sql("""
CREATE TABLE <tu_esquema>.tabla_de_prueba (id INTEGER);
""")

sql("""
DROP TABLE IF EXISTS <tu_esquema>.____________;
""")

In [ ]:
# Verificar que tabla_de_prueba ya no existe
sql("""
SELECT table_name
FROM   information_schema.tables
WHERE  table_schema = '<tu_esquema>'
ORDER  BY table_name;
""")

Hasta aquí escribimos `TU_ESQUEMA` en cada consulta.
Con `SET search_path` el motor busca primero en nuestro schema y no hace falta calificar cada nombre de tabla.

In [ ]:
# A partir de aquí podemos escribir solo el nombre de la tabla
sql("SET search_path TO <tu_esquema>;")

## Ejercicio 2 - DDL: FOREIGN KEY y restricciones nombradas

Creamos `notas` con una llave foránea que referencia a `alumnos`.
`ON DELETE CASCADE` garantiza que al borrar un alumno, sus notas se eliminan automáticamente.

> **Nota sobre `public.viaje.cuotas_pagadas`:** esa columna acepta texto libre,
> por eso encontramos valores como `'si'`, `'SI'`, `'pagado'`, etc.
> Un `CHECK` al momento de crear la tabla hubiera forzado un conjunto fijo de valores
> y evitado ese problema desde el origen.

In [ ]:
# Eliminar la versión sin restricciones del ejercicio 1
sql("DROP TABLE IF EXISTS notas;")

# Recrear notas con FK y CHECK
sql("""
CREATE TABLE notas (
    cod_estudiante  _______       NOT NULL REFERENCES alumnos (cod_estudiante) ON DELETE CASCADE,
    nombre          TEXT,
    materia         TEXT          ___________,
    nota            NUMERIC       _______________
);
""")

La misma tabla con restricciones nombradas explícitamente — forma alternativa equivalente:

```sql
CREATE TABLE notas (
    cod_estudiante  INTEGER,
    nombre          TEXT,
    materia         TEXT,
    nota            NUMERIC,

    CONSTRAINT notas_cod_estudiante_fk
        FOREIGN KEY (cod_estudiante)
        REFERENCES alumnos (cod_estudiante)
        ON DELETE CASCADE,
    CONSTRAINT notas_cod_estudiante_not_null
        CHECK (cod_estudiante IS NOT NULL),
    CONSTRAINT notas_materia_not_null
        CHECK (materia IS ___________),
    CONSTRAINT notas_nota_rango
        _______________
);
```

Nombrar los constraints facilita leer los mensajes de error cuando se viola una restricción.

In [ ]:
# Verificar restricciones de notas
sql("""
SELECT constraint_name,
       constraint_type
FROM   information_schema.table_constraints
WHERE  table_schema = '<tu_esquema>'
  AND  table_name   = 'notas'
ORDER  BY constraint_type;
""")

## Ejercicio 3 - DML: INSERT, UPDATE, DELETE

Insertamos filas una a una, copiamos datos desde `public.alumnos` con `INSERT ... SELECT`,
practicamos `UPDATE` y `DELETE` con y sin `WHERE`, y verificamos el `CASCADE` al borrar un alumno.

In [ ]:
# INSERT fila a fila en alumnos
sql("""
INSERT INTO alumnos (cod_estudiante, nombre, sexo, edad, pais, ciudad,
                     altura, paralelo, anio_nac, mes_nac, dia_nac)
VALUES (1, 'Ana Torres', 'f', 28, 'Ecuador', 'Quito', 165, 'A', 1996, 3, 12);
""")

sql("""
INSERT INTO alumnos (cod_estudiante, nombre, sexo, edad, pais, ciudad,
                     altura, paralelo, anio_nac, mes_nac, dia_nac)
VALUES (2, __________, ___, _____, ______, ______, ______, _____, ______, _____, _____);
""")

sql("""
INSERT INTO alumnos (cod_estudiante, nombre, sexo, edad, pais, ciudad,
                     altura, paralelo, anio_nac, mes_nac, dia_nac)
VALUES (3, 'María León', 'f', 26, 'Perú', 'Lima', 160, 'A', 1998, 11, 22);
""")

In [ ]:
# Verificar las filas insertadas
sql("SELECT * FROM alumnos;")

In [ ]:
# Intentar violar las restricciones: nombre NULL, sexo invalido y edad fuera de rango
# Resultado esperado: ERROR de restriccion
sql("""
INSERT INTO alumnos (cod_estudiante, nombre, sexo, edad, pais, ciudad,
                     altura, paralelo, anio_nac, mes_nac, dia_nac)
VALUES (3, NULL, 'MUJER', 260, 'Perú', 'Lima', 160, 'A', 1998, 11, 22);
""")

In [ ]:
# INSERT en notas
sql("""
INSERT INTO notas (cod_estudiante, nombre, materia, nota)
VALUES (1, 'Ana Torres', 'matematicas', 8.5);
""")

sql("""
INSERT INTO notas (cod_estudiante, nombre, materia, nota)
VALUES (1, 'Ana Torres', 'castellano', 7.0);
""")

sql("""
INSERT INTO notas (cod_estudiante, nombre, materia, nota)
VALUES (2, 'Luis Parra', 'matematicas', 6.5);
""")

In [ ]:
# Intentar insertar una nota de un alumno inexistente
# Resultado esperado: ERROR de integridad referencial
sql("""
INSERT INTO notas (cod_estudiante, nombre, materia, nota)
VALUES (999, 'Fantasma', 'matematicas', 5.0);
""")

In [ ]:
# TRUNCATE: vaciar ambas tablas antes del INSERT desde SELECT
sql("TRUNCATE TABLE ________ ________;")  

# Verificar que ambas tablas quedaron vacías
print(sql("SELECT * FROM alumnos;"))
print(sql("SELECT * FROM notas;"))

In [ ]:
# INSERT desde SELECT: copiar todos los alumnos desde public
sql("""
INSERT INTO alumnos (cod_estudiante, nombre, sexo, edad, pais, ciudad,
                     altura, paralelo, anio_nac, mes_nac, dia_nac)
SELECT ________, nombre, sexo, edad, pais, ciudad,
       altura, paralelo, anio_nac, mes_nac, ________
FROM   ______._______;
""")

In [ ]:
# Verificar cuántos alumnos tenemos
sql("SELECT COUNT(*) AS total FROM alumnos;")

In [ ]:
# UPDATE con WHERE: corregir la ciudad del alumno 1
sql("""
UPDATE _______
SET    ciudad = ________
WHERE  cod_estudiante = ____;
""")

sql("SELECT cod_estudiante, nombre, ciudad FROM alumnos WHERE cod_estudiante = 1;")

In [ ]:
# UPDATE sin WHERE: afecta TODAS las filas
# Descomentar las cuatro líneas para ver el efecto y revertirlo
# conn.autocommit = False
# sql("UPDATE alumnos SET paralelo = 'Z';")
# print(sql("SELECT cod_estudiante, nombre, paralelo FROM alumnos;"))
# conn.rollback(); conn.autocommit = True

> `conn.autocommit = False` abre una transacción explícita. `conn.rollback()` deshace todos los cambios
> desde el último `BEGIN`, como si el `UPDATE` nunca hubiera ocurrido.
> El equivalente en DBeaver es `BEGIN; ... ROLLBACK;`.

In [ ]:
# DELETE con WHERE: insertar fila de prueba y borrarla
sql("""
INSERT INTO alumnos (cod_estudiante, nombre, sexo, edad, pais, ciudad,
                     altura, paralelo, anio_nac, mes_nac, dia_nac)
VALUES (500, 'Para Borrar', 'm', 20, 'Ecuador', 'Quito', 170, 'A', 2004, 1, 1);
""")

print(sql("SELECT * FROM alumnos WHERE cod_estudiante = 500;"))

sql("DELETE FROM alumnos WHERE _________ _ _____;")  

print(sql("SELECT * FROM alumnos WHERE cod_estudiante = 500;"))

In [ ]:
# Demostrar ON DELETE CASCADE
# Ver notas del alumno 1 antes de borrarlo
print(sql("SELECT * FROM notas WHERE cod_estudiante = 1;"))

# Borrar el alumno
sql("DELETE FROM alumnos WHERE cod_estudiante = 1;")

# Las notas del alumno 1 ya no existen
print(sql("SELECT * FROM notas WHERE cod_estudiante = 1;"))

# El resto de notas sigue intacto
sql("SELECT * FROM notas;")

In [ ]:
# DELETE sin WHERE: elimina TODAS las filas
# No ejecutar sin respaldo
# sql("DELETE FROM alumnos;")